# CosMx cell typing — label transfer from Salcher LUAD atlas

**Goal:** Assign cell type labels to CosMx cells using decoupler-based activity
scoring against marker gene sets derived from the Salcher et al. LUAD primary
tumor atlas. Leiden clusters are assigned a cell type label when >30% of cells
in that cluster share the same top-scoring cell type. Unresolved clusters are
labeled `Unidentified`.

**Input:** `outputs/processed/cosmx_umap.h5ad` — CosMx cells with UMAP and Leiden
clusters; `outputs/processed/salcher_cosmx_gene_set.h5ad` — Salcher atlas subset
to CosMx gene set with pseudo-probes for multi-gene probes.

**Output:** `outputs/processed/tumor_data_scored.h5ad` — CosMx AnnData with
`cell_type_assigned` in obs.

## Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in CosMx/scRNA UMAPs

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
cosmx_umap_path    = repo_root / cfg['outputs']['processed'] / 'cosmx_umap.h5ad'
salcher_ref_path   = Path(cfg['datasets']['salcher_atlas']['cosmx_subset'])

# outputs
processed_out = repo_root / cfg['outputs']['processed']
fig_dir       = repo_root / cfg['outputs']['figures']
table_dir     = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir  / 'figure3'
supp_out  = fig_dir  / 'figure3-supp'
table_out = table_dir / 'figure3'

processed_out.mkdir(parents=True, exist_ok=True)
fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = supp_out

## Load data

The CosMx AnnData (QC-filtered, tonsil-excluded, UMAP-embedded, and Leiden-clustered
from `umap_cosmx`) and the Salcher et al. pan-cancer scRNA-seq atlas are loaded.
The Salcher reference has been pre-processed in `prepare_salcher_cosmx_reference`
to retain only the 975 genes present in the CosMx panel, with multi-gene probe
expression computed as the sum of raw counts across individual gene targets.
The reference is further subset here to LUAD primary tumor samples only — ensuring
marker gene signatures reflect the tumor microenvironment biology relevant to this study.

In [ ]:
# load CosMx data with UMAP and Leiden clusters
tumor = ad.read_h5ad(cosmx_umap_path)
print(f"CosMx: {tumor.n_obs:,} cells × {tumor.n_vars} genes")

# load Salcher reference — subset to CosMx gene set with pseudo-probes
adata_ref = ad.read_h5ad(salcher_ref_path)
print(f"Salcher reference: {adata_ref.n_obs:,} cells × {adata_ref.n_vars} genes")

# subset reference to LUAD primary tumor only
adata_ref = adata_ref[adata_ref.obs['disease'] == 'lung adenocarcinoma'].copy()
tumor_ref  = adata_ref[adata_ref.obs['origin'] == 'tumor_primary'].copy()

print(f"\nLUAD primary tumor reference: {tumor_ref.n_obs:,} cells")
print(tumor_ref.obs['ann_coarse'].value_counts())

## Compute marker genes

Marker genes for each coarse cell type are identified from the LUAD primary tumor
reference using a Wilcoxon rank-sum test. The top 20 markers per cell type are
used to score CosMx cells; the top 5 are used for validation dotplots.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sc.tl.rank_genes_groups(tumor_ref, groupby='ann_coarse', method='wilcoxon')

# top 20 markers per cell type — used for scoring
result = tumor_ref.uns['rank_genes_groups']
groups = result['names'].dtype.names

top_genes_dict = {}
for group in groups:
    top_genes_dict[group] = list(result['names'][group][:20])

# top 5 markers per cell type — used for dotplots
marker_genes_dict_dp = {}
for group in groups:
    marker_genes_dict_dp[group] = list(result['names'][group][:5])
marker_genes_dict_dp['MHC I'] = ['MHC I']

print("Marker genes computed for:")
print(list(top_genes_dict.keys()))

In [ ]:
# curated short marker list for validation dotplots
marker_genes_dict_dp_short = {
    'Epithelial cell':      ['KRT18', 'KRT8', 'KRT19', 'EPCAM', 'CLDN4'],
    'Macrophage/Monocyte':  ['TYROBP', 'FCER1G', 'LYZ', 'AIF1', 'HLA-DRA'],
    'Plasma cell':          ['MZB1', 'JCHAIN', 'IGKC', 'XBP1', 'FKBP11'],
    'Stromal':              ['CALD1', 'IGFBP7', 'COL1A2', 'DCN', 'BGN'],
}

# full marker dict for leiden-level dotplot
n_markers = 5
marker_genes_dict_dp = {
    group: sc.get.rank_genes_groups_df(
        tumor_ref, group=group, key='rank_genes_groups'  # default key
    )['names'][:n_markers].tolist()
    for group in tumor_ref.obs['ann_coarse'].unique()
}

## Reference composition

Cell type composition of the LUAD primary tumor reference is shown as a dotplot
to confirm marker gene quality before scoring CosMx cells.

In [ ]:
# dotplot of top 5 markers per cell type in reference
sc.tl.dendrogram(tumor_ref, groupby='ann_coarse')
sc.pl.dotplot(
    tumor_ref, marker_genes_dict_dp,
    groupby='ann_coarse',
    dendrogram=True, use_raw=False,
    figsize=(23, 4), log=True,
)

## Score CosMx cells

Each CosMx cell is scored for each cell type by summing the normalized expression
of the top 20 marker genes for that cell type. The cell type with the highest
summed score is recorded in `max_score_col`. This simple approach is robust to
the sparse gene panel of CosMx and avoids assumptions about gene set distributions.

In [ ]:
%%time

# sum marker gene expression per cell type for each CosMx cell
scores = {}
for cell_type, genes in top_genes_dict.items():
    # find genes present in CosMx panel
    gene_indices = [tumor.var_names.get_loc(g) for g in genes if g in tumor.var_names]
    if len(gene_indices) == 0:
        print(f"Warning: no genes found for '{cell_type}'")
        continue

    expr = tumor.X[:, gene_indices]
    if hasattr(expr, 'toarray'):
        expr = expr.toarray()

    scores[f'{cell_type}_score'] = expr.sum(axis=1).flatten()

scores_df = pd.DataFrame(scores, index=tumor.obs.index)
tumor.obs = tumor.obs.join(scores_df)

# record top-scoring cell type per cell
score_cols = list(scores_df.columns)
tumor.obs['max_score_col'] = tumor.obs[score_cols].idxmax(axis=1)
print(tumor.obs['max_score_col'].value_counts())

## Assign cell types by cluster

A Leiden cluster is assigned a specific cell type label when more than 30% of its
cells share the same top-scoring cell type. This cluster-level approach is more
robust than per-cell assignment given the sparse CosMx gene panel.

Unresolved clusters — where no single cell type reaches the 30% threshold — are
labeled `Unidentified`. Extensive attempts to further subtype these clusters into
specific immune populations were not successful given the limited immune gene
coverage of the CosMx panel.

In [ ]:
# epithelial clusters — >30% of cells have Epithelial cell as top score
tumor.obs['epithelial_cluster'] = tumor.obs.leiden.map(
    tumor[tumor.obs.max_score_col == 'Epithelial cell_score'].obs['leiden'].value_counts()
    / tumor.obs['leiden'].value_counts() > 0.3
)

# assign cell types — start with Unidentified, overwrite with specific labels
# order matters: later assignments take precedence
tumor.obs['cell_type_assigned'] = 'Unidentified'

tumor.obs.loc[
    tumor.obs['epithelial_cluster'] == True, 'cell_type_assigned'
] = 'Epithelial'

tumor.obs.loc[
    tumor.obs.leiden.map(
        tumor[tumor.obs.max_score_col == 'Stromal_score'].obs['leiden'].value_counts()
        / tumor.obs['leiden'].value_counts() > 0.3
    ) == True, 'cell_type_assigned'
] = 'Stromal'

tumor.obs.loc[
    tumor.obs.leiden.map(
        tumor[tumor.obs.max_score_col == 'Plasma cell_score'].obs['leiden'].value_counts()
        / tumor.obs['leiden'].value_counts() > 0.3
    ) == True, 'cell_type_assigned'
] = 'Plasma'

tumor.obs.loc[
    tumor.obs.leiden.map(
        tumor[tumor.obs.max_score_col == 'Macrophage/Monocyte_score'].obs['leiden'].value_counts()
        / tumor.obs['leiden'].value_counts() > 0.3
    ) == True, 'cell_type_assigned'
] = 'Macrophage/Monocyte'

print(tumor.obs['cell_type_assigned'].value_counts())

In [ ]:
# rename Macrophage/Monocyte to Macrophage-Monocyte to avoid forward slash
# rename for display
tumor.obs['Assigned Cell Type'] = tumor.obs['cell_type_assigned']


# in h5ad column names (forward slashes will be disallowed in future anndata versions)
tumor.obs['cell_type_assigned'] = tumor.obs['cell_type_assigned'].replace(
    'Macrophage/Monocyte', 'Macrophage-Monocyte'
)
tumor.obs['Assigned Cell Type'] = tumor.obs['Assigned Cell Type'].replace(
    'Macrophage/Monocyte', 'Macrophage-Monocyte'
)

## Validation plots

UMAP colored by assigned cell type and dotplots of canonical marker gene expression
confirm that cluster labels are biologically coherent. The `max_score_col` UMAP
shows the raw per-cell scoring before cluster-level assignment.

In [ ]:
sc.settings.figdir = supp_out

# UMAP by max score — raw per-cell scoring
sc.pl.umap(tumor, color='max_score_col', title='Top Scoring Cell Type per Cell',
 save='_umap_max_score.pdf')

In [ ]:
celltype_palette = {
    'Epithelial':          '#E41A1C',  # red
    'Macrophage-Monocyte': '#4DAF4A',  # green
    'Plasma':              '#FF7F00',  # orange
    'Stromal':             '#A65628',  # brown
    'Unidentified':        '#984EA3',  # purple
}

sc.pl.umap(tumor, color='Assigned Cell Type',
           palette=celltype_palette,
           title='Assigned Cell Type',
           save='_figS3a_celltypes.pdf')



In [ ]:
# note: scanpy raises a warning that groupby categories and var_group_labels differ
# because the marker dict keys are used as column group labels, not matched to
# groupby categories. this is expected behavior and does not affect the plot.

# dotplot — curated short markers per assigned cell type
marker_genes_dict_dp_short = {
    'Epithelial':          ['KRT18', 'KRT8', 'KRT19', 'EPCAM', 'CLDN4'],
    'Macrophage-Monocyte': ['TYROBP', 'FCER1G', 'LYZ', 'AIF1', 'HLA-DRA'],
    'Plasma':              ['MZB1', 'JCHAIN', 'IGKC', 'XBP1', 'FKBP11'],
    'Stromal':             ['CALD1', 'IGFBP7', 'COL1A2', 'DCN', 'BGN'],
}

# dotplot — curated short markers per assigned cell type
with plt.rc_context({'font.size': 16}):
    sc.pl.dotplot(
        tumor, marker_genes_dict_dp_short,
        groupby='Assigned Cell Type',
        dendrogram=True, use_raw=False,
        show=False,
    )
    fig = plt.gcf()
    fig.set_size_inches(16, 5)
    plt.subplots_adjust(right=0.72)
    plt.savefig(supp_out / 'figS3_celltype_markers_dotplot.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# dotplot — full marker dict by leiden cluster
# note: same warning as above about groupby categories vs var_group_labels — expected
with plt.rc_context({'font.size': 12}):
    sc.pl.dotplot(
        tumor, marker_genes_dict_dp,
        groupby='leiden',
        dendrogram=True, use_raw=False,
        show=False,
    )
    fig = plt.gcf()
    fig.set_size_inches(22, 10)
    plt.subplots_adjust(right=0.72)
    plt.savefig(supp_out / 'figS3_leiden_markers_dotplot.pdf', bbox_inches='tight')
    plt.show()

### Leiden cluster to cell type mapping

| Leiden clusters | Assigned cell type | Basis |
|---|---|---|
| 3, 5, 15, 18, 19, 2, 21, 13, 24, 14, 23, 16, 20, 12 | Epithelial | KRT18, KRT19, EPCAM, CLDN4 enrichment |
| 27 | Macrophage-Monocyte | LYZ, FCER1G, AIF1, HLA-DRA enrichment |
| 8, 17, 9, 22 | Plasma | IGKC, JCHAIN, MZB1, XBP1 enrichment |
| 4, 6 | Stromal | BGN, DCN, COL1A2, IGFBP7 enrichment |
| 26 | Unidentified (B cell/cDC signal) | Diffuse B cell + cDC markers, below 30% threshold |
| 0, 1, 7, 10, 11, 25 | Unidentified | No dominant cell type signal |

Note: assignments are based on >30% of cells in a cluster having a given cell type
as their top marker gene score. Clusters not meeting this threshold for any specific
type are labeled Unidentified.

## Save

The scored and labeled AnnData is saved to `outputs/processed/`. This file is
the primary input for `mhc2_ihc_deg` and `ihc_groups_cell_type_enrichment`.

In [ ]:
out_path = processed_out / 'tumor_data_scored.h5ad'
tumor.write(out_path)
print(f"Saved {tumor.n_obs:,} cells → {out_path}")